# 墨阑 1,276 本真人小说模型 —— Google Colab 免费版 T4 一键训练工作台 (v2)

> **定位**：专为 **Google Colab 免费版 T4 GPU（16GB 显存 / ~12.7GB 内存）** 深度加固的 Qwen2.5-7B-Instruct 4-bit QLoRA 训练流水线。
> **数据**：`molan-home/data/lora-training/` v2 零假模板真人小说数据集（**2,135 条训练 + 117 条书本级隔离验证**，70% 真实前文续写 + 30% 干净节拍写作，单条 ≤ 2,176 字零截断，压缩后仅 ~11 MB）。

### 核心加固特性（专治 Colab 免费版痛点）
1. **Google Drive 优先读数**：只需把 `train_sharegpt.jsonl`（~11 MB）与 `val_sharegpt.jsonl`（~0.6 MB）上传一次到云盘 `MyDrive/molan-lora/data/`，断线重连无需重传。
2. **自动断点续训**：每 50 步自动把 Checkpoint 持久化到 Google Drive；若免费 T4 中途断线，重连 GPU 后直接从头再点一次运行即可自动接续。
3. **仅对小说正文算 Loss (`train_on_responses_only`)**：屏蔽 system/user 提示词，模型全部算力集中学习真人网文语感、对话节奏与白描细节。
4. **书本级验证集实时监控**：每 50 步评估独立 63 本验证集 Loss，防止过拟合死记硬背。
5. **T4 防内存爆炸（Anti-OOM）分步导出**：训练完立刻先备份轻量 LoRA（~150 MB）到云盘，再释放显存/内存并用低内存模式导出 Ollama 专用 `q4_k_m.gguf`；并附带 **Cell 10 独立重启恢复导出单元**，彻底告别 12.7GB RAM 崩溃。

### 上手 4 步走
1. 顶部菜单点击 **代码执行程序 (Runtime) -> 更改运行时类型 (Change runtime type) -> T4 GPU -> 保存**。
2. 建议先在 Google Drive 新建文件夹 `molan-lora/data/`，把本地 `molan-home/data/lora-training/train_sharegpt.jsonl` 和 `val_sharegpt.jsonl` 拖进去（也可跳过此步，在 Cell 3 网页直接上传并自动转存云盘）。
3. 按顺序运行 **Cell 1 ~ Cell 9**（2 Epochs ≈ 267 步，免费版 T4 全程约 **35~50 分钟**）。
4. 从 Google Drive `molan-lora/gguf/` 下载 `qwen2.5-7b-molan-q4_k_m.gguf`（约 4.5 GB），配合本地 `molan-home/training/Modelfile` 执行 `ollama create molan-humanizer-7b -f Modelfile` 即可免费离线写小说。

> ⚠️ **合规声明**：训练语料来自个人本地 `资源库/小说原本`，微调所得权重与数据集仅限个人学习与本地创作使用，请勿公开上传或分发。

In [ ]:
# === Cell 1: 检查 GPU/内存环境 & 挂载 Google Drive ===
import os
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ 未检测到 GPU！请点击顶部菜单：代码执行程序 (Runtime) -> 更改运行时类型 -> 选择 T4 GPU 后重试。"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
ram_gb = 0.0
if os.path.exists("/proc/meminfo"):
    with open("/proc/meminfo", "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("MemTotal:"):
                ram_gb = int(line.split()[1]) / (1024 ** 2)
                break

print(f"✅ GPU 就绪: {gpu_name} | 显存: {vram_gb:.1f} GB | 系统内存: {ram_gb:.1f} GB")
if ram_gb and ram_gb < 16:
    print("ℹ️ 检测到免费版标准内存 (~12.7GB)，已自动启用防 OOM 分步保存与低内存 GGUF 导出策略。")

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/molan-lora"
DATA_DIR = os.path.join(DRIVE_ROOT, "data")
OUT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
LORA_DIR = os.path.join(DRIVE_ROOT, "lora_adapter")
GGUF_DIR = os.path.join(DRIVE_ROOT, "gguf")

for d in (DATA_DIR, OUT_DIR, LORA_DIR, GGUF_DIR):
    os.makedirs(d, exist_ok=True)

print("📁 Google Drive 工作目录已就绪:")
print(f"   - 数据目录: {DATA_DIR}")
print(f"   - 断点目录: {OUT_DIR}")
print(f"   - LoRA备份: {LORA_DIR}")
print(f"   - GGUF导出: {GGUF_DIR}")

In [ ]:
# === Cell 2: 安装 Unsloth 及训练依赖（Colab T4 极速兼容版，约 1~2 分钟）===
import os

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -q unsloth
else:
    !pip install -q --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
    !pip install -q sentencepiece protobuf "datasets>=3.0.0,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install -q --no-deps unsloth

import unsloth, transformers, trl, peft
print(
    f"✅ 依赖就绪 | unsloth={getattr(unsloth, '__version__', 'installed')} "
    f"| torch={torch.__version__} | transformers={transformers.__version__} "
    f"| trl={trl.__version__} | peft={peft.__version__}"
)

In [ ]:
# === Cell 3: 加载或上传数据集（优先读取 Google Drive，一次上传永久复用）===
import os
import shutil
from google.colab import files

TRAIN_FILE = os.path.join(DATA_DIR, "train_sharegpt.jsonl")
VAL_FILE = os.path.join(DATA_DIR, "val_sharegpt.jsonl")

missing = [p for p in (TRAIN_FILE, VAL_FILE) if not os.path.exists(p)]
if missing:
    print("⚠️ Google Drive 中尚未发现数据集，请在下方弹窗选择本地文件上传：")
    print("   路径: molan-home/data/lora-training/train_sharegpt.jsonl 和 val_sharegpt.jsonl")
    print("   （上传后会自动保存到 Google Drive，下次断线重连无需再传）")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dst = os.path.join(DATA_DIR, os.path.basename(fname))
        with open(dst, "wb") as f:
            f.write(data)
        print(f"   已转存到云盘: {dst} ({len(data) / 1048576:.2f} MB)")

assert os.path.exists(TRAIN_FILE), f"❌ 缺少训练集: {TRAIN_FILE}"
assert os.path.exists(VAL_FILE), f"❌ 缺少验证集: {VAL_FILE}"

with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    train_lines = sum(1 for line in f if line.strip())
with open(VAL_FILE, "r", encoding="utf-8") as f:
    val_lines = sum(1 for line in f if line.strip())

print(f"✅ 数据集就绪（来源 Google Drive）:")
print(f"   - 训练集: {train_lines} 条 ({os.path.getsize(TRAIN_FILE) / 1048576:.2f} MB)")
print(f"   - 验证集: {val_lines} 条 ({os.path.getsize(VAL_FILE) / 1048576:.2f} MB)")

In [ ]:
# === Cell 4: 加载 Qwen2.5-7B-Instruct (4-bit) 并注入 LoRA ===
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048  # v2 数据集最大 2,176 汉字 (约 1,550 tokens)，2048 窗口零截断且最省显存
LORA_R = 32            # 2,135 样本配 r=32 / alpha=64 兼顾文风迁移力与防过拟合
LORA_ALPHA = 64

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    use_rslora=True,
    random_state=2026,
)

model.print_trainable_parameters()

In [ ]:
# === Cell 5: 将 ShareGPT/ChatML 数据集渲染为 Qwen2.5 训练序列 ===
from datasets import load_dataset

raw_datasets = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE},
)

def format_to_chatml(batch):
    texts = [
        tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        for msgs in batch["messages"]
    ]
    return {"text": texts}

train_dataset = raw_datasets["train"].map(
    format_to_chatml,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="格式化训练集",
)
val_dataset = raw_datasets["validation"].map(
    format_to_chatml,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
    desc="格式化验证集",
)

sample_text = train_dataset[0]["text"]
assert "<|im_start|>user\n" in sample_text and "<|im_start|>assistant\n" in sample_text, (
    "❌ ChatML 模板异常，请检查 tokenizer 配置。"
)
sample_tokens = len(tokenizer(sample_text)["input_ids"])
print(f"✅ 格式化完成 | 训练集: {len(train_dataset)} 条 | 验证集: {len(val_dataset)} 条")
print(f"   首条样本 Token 数: {sample_tokens} / {MAX_SEQ_LENGTH}")
print("   首条样本预览:")
print(sample_text[:360] + "\n...")

In [ ]:
# === Cell 6: 配置 SFTTrainer + 仅正文算 Loss + 验证集监控 + T4/L4/A100 自适应精度 ===
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

EPOCHS = 2           # 2,135 条 × 2 Epochs ≈ 267 步（免费 T4 约 35~50 分钟）
BATCH_SIZE = 2       # T4 16GB 安全批大小
GRAD_ACCUM = 8       # 等效批大小 = 16
LEARNING_RATE = 1.5e-4

use_bf16 = is_bfloat16_supported()
print(f"⚙️ 混合精度模式: {'bf16 (L4/A100)' if use_bf16 else 'fp16 (T4)'}")

training_args = SFTConfig(
    output_dir=OUT_DIR,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.01,
    fp16=not use_bf16,
    bf16=use_bf16,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=2026,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

# 关键优化：屏蔽 system/user 提示词，仅在 assistant 输出的真人小说正文上计算梯度
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# 抽检屏蔽效果：确保 labels 中既有 -100（提示词）又有真实正文 token
sample_labels = trainer.train_dataset[0]["labels"]
masked_cnt = sum(1 for x in sample_labels if x == -100)
active_cnt = len(sample_labels) - masked_cnt
assert active_cnt > 50, "❌ 正文有效训练 Token 过少，请检查 ChatML 分隔符！"
print(f"✅ train_on_responses_only 生效 | 提示词屏蔽: {masked_cnt} tokens | 正文学习: {active_cnt} tokens")

In [ ]:
# === Cell 7: 启动训练（自动检测 Google Drive 断点续训 + 完训即刻备份 LoRA）===
import glob
import re

def find_latest_checkpoint(out_dir):
    ckpts = glob.glob(os.path.join(out_dir, "checkpoint-*"))
    if not ckpts:
        return None
    def step_num(path):
        m = re.search(r"checkpoint-(\d+)$", path)
        return int(m.group(1)) if m else -1
    ckpts.sort(key=step_num)
    return ckpts[-1]

latest_ckpt = find_latest_checkpoint(OUT_DIR)
if latest_ckpt:
    print(f"🔄 检测到历史断点，自动续训: {latest_ckpt}")
    trainer_stats = trainer.train(resume_from_checkpoint=latest_ckpt)
else:
    print("🚀 开始全新训练（每 50 步自动保存断点至 Google Drive）...")
    trainer_stats = trainer.train()

# 完训第一时间备份轻量化 LoRA 权重到云盘（~150MB，绝对不会 OOM）
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

mins = trainer_stats.metrics.get("train_runtime", 0) / 60.0
final_loss = trainer_stats.metrics.get("train_loss", 0.0)
print(f"\n🎉 训练完成！耗时: {mins:.1f} 分钟 | 最终 Train Loss: {final_loss:.4f}")
print(f"💾 LoRA 权重已安全备份至 Google Drive: {LORA_DIR}")

In [ ]:
# === Cell 8: 真实 ChatML 推理试写验收（前文续写 + 节拍写作）===
FastLanguageModel.for_inference(model)

def test_novel_gen(system_prompt, user_prompt, max_new_tokens=420):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.92,
        top_p=0.92,
        repetition_penalty=1.08,
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

sys_cont = (
    "你是一名擅长写高质感中文网络小说的资深作者。\n"
    "请紧接用户给出的前文片段自然续写正文，保持人物性格、视口连续与口语白描节奏。\n"
    "写作准则：以具体动作、细节和对白推动情节；段落轻快有呼吸感；禁止空洞总结与机械AI套话。"
)
user_cont = (
    "【题材】东方仙侠\n"
    "【字数目标】约 400 字\n"
    "【前文片段】\n"
    "林凡蹲在药田里拔草，才拔了半亩，腰已经酸得直不起来了。\n"
    "他刚把草堆放好，赵师兄就带着两个跟班过来了。\n"
    "“林凡，这个月交的灵草成色不够。”赵师兄说道，“你的月俸要扣一半。”\n"
    "林凡抬起头，问道：“我交的都是照数采的，为什么不够？”\n\n"
    "请紧接上述前文自然续写接下来的情节正文，直接输出小说正文，不要加标题或说明。"
)

print("=== 试写验收：真实前文续写 ===")
print(test_novel_gen(sys_cont, user_cont))

In [ ]:
# === Cell 9: 防 OOM 安全导出为 Ollama 专用 GGUF (q4_k_m, 约 4.5GB) ===
# 先彻底清理训练器与数据集占用的内存，再用低内存限制导出 GGUF
import gc
import glob
import shutil
import torch

for var_name in ("trainer", "train_dataset", "val_dataset", "raw_datasets"):
    if var_name in globals():
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

LOCAL_GGUF_WORK = "/content/molan_gguf_export"
os.makedirs(LOCAL_GGUF_WORK, exist_ok=True)

print("⚙️ 开始合并 LoRA 并量化导出 GGUF (q4_k_m)，约需 5~10 分钟...")
print("   （若免费版 12.7GB 内存在此步触发系统重启，请放心：LoRA 已在 Cell 7 存好云盘，直接重启后仅运行 Cell 10 即可！）")

model.save_pretrained_gguf(
    LOCAL_GGUF_WORK,
    tokenizer,
    quantization_method="q4_k_m",
    maximum_memory_usage=0.4,
)

# 查找生成的 .gguf 并统一重命名复制到 Google Drive
gguf_candidates = sorted(
    glob.glob(os.path.join(LOCAL_GGUF_WORK, "*.gguf"))
    + glob.glob("/content/*q4_k_m*.gguf")
    + glob.glob("/content/molan_gguf_export*/*.gguf")
)
assert gguf_candidates, "❌ 未找到生成的 GGUF 文件，请检查上方输出日志。"

src_gguf = [p for p in gguf_candidates if "q4_k_m" in os.path.basename(p).lower()] or gguf_candidates
target_gguf = os.path.join(GGUF_DIR, "qwen2.5-7b-molan-q4_k_m.gguf")
shutil.copy2(src_gguf[-1], target_gguf)

size_gb = os.path.getsize(target_gguf) / (1024 ** 3)
print(f"\n🎉 GGUF 导出并同步 Google Drive 成功！")
print(f"   云盘路径: {target_gguf} ({size_gb:.2f} GB)")
print("   下载到本地后，与 molan-home/training/Modelfile 放同一目录，执行：")
print("   ollama create molan-humanizer-7b -f Modelfile")

--- 
### 🆘 Cell 10（备用防崩溃通道）：仅当 Cell 9 导出 GGUF 时因免费版 12.7GB 内存不足重启才需要运行
如果 Cell 9 已经顺利导出 GGUF，**不需要运行下面这个单元**。
如果 Cell 9 崩溃了：先点击顶部 **代码执行程序 -> 重新启动会话 (Restart session)**（把 12.7GB 内存彻底清零），然后**只运行 Cell 10**（它会直接加载云盘里已经保存好的 LoRA 权重并低内存导出 GGUF）。

In [ ]:
# === Cell 10 (备用): 纯净内存恢复导出 GGUF（仅在 Cell 9 OOM 重启后单独运行）===
import gc
import glob
import os
import shutil
import torch
from google.colab import drive
from unsloth import FastLanguageModel

drive.mount("/content/drive")
LORA_DIR = "/content/drive/MyDrive/molan-lora/lora_adapter"
GGUF_DIR = "/content/drive/MyDrive/molan-lora/gguf"
os.makedirs(GGUF_DIR, exist_ok=True)

assert os.path.exists(LORA_DIR), f"❌ 未找到已保存的 LoRA 目录: {LORA_DIR}"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

LOCAL_GGUF_WORK = "/content/molan_gguf_export"
os.makedirs(LOCAL_GGUF_WORK, exist_ok=True)

gc.collect()
torch.cuda.empty_cache()

model.save_pretrained_gguf(
    LOCAL_GGUF_WORK,
    tokenizer,
    quantization_method="q4_k_m",
    maximum_memory_usage=0.35,
)

gguf_candidates = sorted(
    glob.glob(os.path.join(LOCAL_GGUF_WORK, "*.gguf"))
    + glob.glob("/content/*q4_k_m*.gguf")
    + glob.glob("/content/molan_gguf_export*/*.gguf")
)
assert gguf_candidates, "❌ 未找到生成的 GGUF 文件。"
src_gguf = [p for p in gguf_candidates if "q4_k_m" in os.path.basename(p).lower()] or gguf_candidates
target_gguf = os.path.join(GGUF_DIR, "qwen2.5-7b-molan-q4_k_m.gguf")
shutil.copy2(src_gguf[-1], target_gguf)
print(f"✅ 恢复导出成功: {target_gguf} ({os.path.getsize(target_gguf) / (1024 ** 3):.2f} GB)")